In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Production layout: add project root and src (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
load_dotenv(_root / ".env")
load_dotenv("/app/.env")
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import datetime

print(f"CloudStorageProvider import OK: {CloudStorageProvider}")

CloudStorageProvider import OK: <class 'storage.cloud.CloudStorage.CloudStorageProvider'>


In [2]:
pg_conn = PgConn("historical")
df = pg_conn.get_stocks_prices()
if df is None:
    raise RuntimeError(
        "get_stocks_prices() failed — check the error printed above "
        "(connection, table name, or schema)."
    )

Connection to the database successful!
Table name set to: historical


In [3]:
df.head()

,ref,book,date,open,high,low,close,adj_close,volume,created_at
0,https://finance.yahoo.com,ada-usd,2026-07-19,0.1666,0.1683,0.1646,0.1665,0.1665,170598480,2026-07-19 15:12:28.513658
1,https://finance.yahoo.com,ada-usd,2026-07-18,0.1666,0.1690,0.1635,0.1666,0.1666,241810290,2026-07-19 15:13:44.797510
2,https://finance.yahoo.com,ada-usd,2026-07-17,0.1607,0.1669,0.1579,0.1666,0.1666,380301500,2026-07-19 15:13:45.200794
3,https://finance.yahoo.com,ada-usd,2026-07-16,0.1651,0.1662,0.1602,0.1607,0.1607,242986450,2026-07-19 15:13:45.363342
4,https://finance.yahoo.com,ada-usd,2026-07-15,0.1652,0.1687,0.1622,0.1651,0.1651,309478890,2026-07-19 15:13:45.495070


In [4]:
df.count()

ref           395
book          395
date          395
open          395
high          395
low           395
close         395
adj_close     395
volume        395
created_at    395
dtype: int64

In [5]:
# Assuming df is your DataFrame
date_column_type = df['date'].dtype
print("Type of values in 'date' column:", date_column_type)

Type of values in 'date' column: object


In [6]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
    
    class Export():
        def __init__(self, dataframe):
            self.cloudProvider = CloudStorageProvider()
            self.df = dataframe
            
        def export_stocks_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_timestamp(self.df, bucket_name, prefix_path, file_format)
            
        def export_stocks_to_s3_full_file(self, bucket_name, prefix_path, filename):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)

        def get_data_csv_file_by_datetime(
            self, bucket_name, prefix_path, year, month, day, book=None, hour=None, minute=None
        ):
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_dataframe_from_specific_datetime(
                bucket_name,
                prefix_path,
                book=book,
                year=year,
                month=month,
                day=day,
                hour=hour,
                minute=minute,
            )
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData(self):
            self.df = pg_conn.get_stocks_prices()
            return self.df
        
        def filter_by_current_date(self):
            today = datetime.today().date()
            dates = pd.to_datetime(self.df["date"], errors="coerce")
            return self.df[dates.dt.date == today]
        
        def filter_by_date_range(self, df, start_date, end_date):
            # Convert start_date and end_date strings to datetime objects
            start_date = pd.to_datetime(start_date)
            end_date = pd.to_datetime(end_date)
            
            df['date'] = pd.to_datetime(df['date'])
            # Filter by date range
            filtered_df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
            return filtered_df
        
        def filter_by_book(self, df, book=None):
            if book is None or (isinstance(book, str) and not book.strip()):
                return df
            return df[df["book"].astype(str).str.strip() == str(book).strip()]
        
        def get_unique_by_date(self, df):
            # Sort the DataFrame by 'date' in descending order
            df_sorted = df.sort_values(by='date', ascending=False)

            # Drop duplicates, keeping only the first occurrence for each unique combination of book and date
            df_unique_latest = df_sorted.drop_duplicates(subset=['book', 'date'])
            df_unique_latest.count()
            return df_unique_latest
            
    class Transform():
        def extractStopWords():
            pass

# Re-run this cell after git pull / Reload Notebook from Disk
import inspect
if "CloudStorage.CloudStorageProvider" in inspect.getsource(DataETL.Export.__init__):
    raise RuntimeError(
        "Stale DataETL code in memory. Use File → Reload Notebook from Disk, "
        "Kernel → Restart, then re-run cells 1–6."
    )

In [7]:
FILTER_BY_CURRENT_DATE = False
FILTER_BY_RANGE_DATE = True
FILTER_BY_BOOK_DATE = False
# TODO 2017-2016
etl = DataETL(df)
etl_process = etl.Process(etl.df)
uniqued_df = etl_process.get_unique_by_date(etl_process.df)
processed_df = uniqued_df

if FILTER_BY_CURRENT_DATE == True:
    processed_df = etl_process.filter_by_current_date(uniqued_df)
    processed_df.head()
elif FILTER_BY_RANGE_DATE == True:
    start_date='2026-07-11'
    end_date='2026-07-19'
    processed_df = etl_process.filter_by_date_range(uniqued_df, start_date, end_date)
    processed_df.head()
elif FILTER_BY_BOOK_DATE == True:
    processed_df = etl_process.filter_by_current_date(uniqued_df)
    processed_df.head()

In [8]:
import inspect

if "CloudStorage.CloudStorageProvider" in inspect.getsource(DataETL.Export.__init__):
    raise RuntimeError(
        "Stale DataETL in kernel. Reload notebook from disk, restart kernel, re-run cell 6."
    )

from config.settings import get_settings

_settings = get_settings()
bucket_name = _settings.aws.stocks_bucket
if not bucket_name:
    raise RuntimeError(
        "No stocks S3 bucket configured. Set AWS_DEFAULT_STOCKS_BUCKET in .env, "
        "then restart Jupyter."
    )
print(f"Using S3 bucket: {bucket_name}")

etl_export = etl.Export(processed_df)
prefix_path = "stocks/crypto"
post_full_csv = False
post_to_s3 = True
now = datetime.now()
filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
file_format = "csv"

if post_to_s3 and not processed_df.empty:
    if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
        raise RuntimeError(
            "AWS credentials not configured. Set AWS_ACCESS_KEY_ID and "
            "AWS_SECRET_ACCESS_KEY in .env, then restart Jupyter."
        )
    etl_export.export_stocks_to_s3(bucket_name, prefix_path, file_format)
if post_full_csv and not processed_df.empty:
    if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
        raise RuntimeError(
            "AWS credentials not configured. Set AWS_ACCESS_KEY_ID and "
            "AWS_SECRET_ACCESS_KEY in .env, then restart Jupyter."
        )
    etl_export.export_stocks_to_s3_full_file(bucket_name, prefix_path, filename)

Using S3 bucket: test-financial-stocks-bucket
Bucket 'test-financial-stocks-bucket' already exists.
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=ada-usd/year=2026/month=07/day=19/format=csv/20260719-ada-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=ondo-usd/year=2026/month=07/day=19/format=csv/20260719-ondo-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=wif-usd/year=2026/month=07/day=19/format=csv/20260719-wif-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=lrc-usd/year=2026/month=07/day=19/format=csv/20260719-lrc-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=crv-usd/year=2026/month=07/day=19/format=csv/20260719-crv-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=bat-usd/year=2026

In [9]:
ingest_data = False
get_full_file = False
get_by_datetime = True
df_from_file = None

if ingest_data:
    etl_ingestion = etl.Ingestion(etl.df)
    _settings = get_settings()
    bucket_name = _settings.aws.stocks_bucket
    if not bucket_name:
        raise RuntimeError(
            "No stocks S3 bucket configured. Set AWS_DEFAULT_STOCKS_BUCKET in .env."
        )
    prefix_path = "stocks/crypto/"
    book = ""  # i.e. "ada-usd" (optional; empty = all books under date)
    year = "2026"
    month = "05"
    day = "24"
    hour = None
    minute = None
    book_arg = book.strip().lower() if book and str(book).strip() else None
    if get_full_file:
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, prefix_path)
    elif get_by_datetime:
        df_from_file = etl_ingestion.get_data_csv_file_by_datetime(
            bucket_name, prefix_path, year, month, day, book=book_arg, hour=hour, minute=minute
        )

In [10]:
if df_from_file is not None:
    print(df_from_file.head())